# Module 6: SHAP and Model Explainability
**ACS Predictive Analytics Curriculum**

Concepts covered:
- Why explainability matters for child welfare
- SHAP values: the math and the intuition
- shapviz package in R
- Per-case waterfall plots (caseworker communication)
- Global importance plots
- Fairness audit using SHAP by borough
- Counterfactual explanations


In [ ]:
install.packages(c('shapviz','treeshap','counterfactuals'),
                 repos='https://cran.rstudio.com/', quiet=TRUE)
library(tidyverse)
library(tidymodels)
library(shapviz)
library(ranger)

scr      <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE)
features <- read_csv('data/acs_features.csv',    show_col_types=FALSE)
cat('Packages and data loaded\n')

## SECTION 6.1: Why Explainability Matters for ACS

In [ ]:
# Without explainability:
# Caseworker: 'This case scored 0.87 - why?'
# Model:      'I cannot tell you.'
# Result:     Caseworker ignores the score. Tool gets abandoned.

# With SHAP:
# Caseworker: 'This case scored 0.87 - why?'
# SHAP:       'Prior reports (+0.31), DV history (+0.18),
#              child under 5 (+0.12), shelter (+0.09)'
# Result:     Caseworker verifies, trusts, acts on the score.

# Epic's sepsis predictor failed because:
# 1. Low precision -> too many false alarms
# 2. No explanation -> nurses couldn't verify or override
# ACS must avoid both mistakes

cat('SHAP explains the MODEL PREDICTION, not ground truth.')
cat('\nIf model is wrong, SHAP faithfully explains why it was wrong.')
cat('\nCaseworker judgment always remains the final decision.')

## SECTION 6.2: Train Model for SHAP Analysis

In [ ]:
set.seed(42)

FEATURES <- c('prior_reports_12mo','prior_substantiated_flag',
               'dv_history_flag','child_age_under_5',
               'shelter_involvement_flag','reporter_accuracy_score',
               'caseworker_caseload','n_children_in_household')

df <- features %>%
  mutate(
    target = as.factor(needs_investigative_consultation),
    across(where(is.logical), as.integer)
  ) %>%
  drop_na(all_of(FEATURES))

train_idx <- sample(nrow(df), 0.8*nrow(df))
train <- df[train_idx,]
test  <- df[-train_idx,]

# Ranger RF: supports SHAP via shapviz
rf_model <- ranger(
  formula   = target ~ .,
  data      = train %>% select(all_of(c(FEATURES, 'target'))),
  num.trees = 200,
  probability = TRUE,  # REQUIRED for SHAP
  importance = 'impurity',
  seed      = 42
)
cat('Model trained\n')

## SECTION 6.3: SHAP Values with shapviz

In [ ]:
# shapviz integrates directly with ranger
# Computes SHAP values using TreeSHAP algorithm
# O(TLD^2) complexity - fast even with many features

X_test <- test %>% select(all_of(FEATURES)) %>% as.data.frame()

# Create shapviz object
shp <- shapviz(rf_model, X_pred=X_test, X=X_test)

# GLOBAL importance: average |SHAP| per feature
sv_importance(shp, kind='bar') +
  labs(title='Global Feature Importance (Mean |SHAP|)',
       subtitle='Average contribution to predictions across all test cases',
       x='Mean |SHAP Value|') +
  theme_minimal(base_size=12)

# NOTE: This is more reliable than tree impurity importance
# because it measures ACTUAL contribution to predictions
# not correlation with tree splits

In [ ]:
# BEESWARM: shows distribution of SHAP values per feature
sv_importance(shp, kind='beeswarm') +
  labs(title='SHAP Value Distribution by Feature',
       subtitle='Each dot = one case | Color = feature value',
       x='SHAP Value (contribution to prediction)') +
  theme_minimal(base_size=12)

# HOW TO READ:
# Red dots on right = high feature value pushes score up
# Blue dots on left = low feature value pushes score down
# Spread = how much this feature varies in its contribution

## SECTION 6.4: Per-Case Waterfall (Caseworker Communication)

In [ ]:
# Pick the highest-scoring case in test set
high_risk_idx <- which.max(predict(rf_model, X_test)$predictions[,'1'])

# Waterfall: shows each feature's contribution for ONE case
sv_waterfall(shp, row_id=high_risk_idx) +
  labs(title=paste('SHAP Explanation for Case',
                   test$report_id[high_risk_idx]),
       subtitle='Each bar = feature contribution | Red = increases risk | Blue = decreases') +
  theme_minimal(base_size=12)

# WHAT CASEWORKER SEES:
# 'This case scored 0.87 because:
#  - 4 prior reports in 12 months: +0.31
#  - Domestic violence history: +0.18
#  - Child under 5 years old: +0.12
#  - Shelter involvement: +0.09
#  - Anonymous reporter: -0.08 (slightly reduces confidence)
#  Starting from baseline of 0.25'

# This is the difference between a tool that gets used
# and a tool that gets ignored

In [ ]:
# Force plot: alternative visualization for single case
sv_force(shp, row_id=high_risk_idx) +
  labs(title='Force Plot: Feature Contributions') +
  theme_minimal()

# Compare: lowest scoring case
low_risk_idx <- which.min(predict(rf_model, X_test)$predictions[,'1'])
sv_waterfall(shp, row_id=low_risk_idx) +
  labs(title=paste('SHAP Explanation: Low Risk Case',
                   test$report_id[low_risk_idx])) +
  theme_minimal(base_size=12)

## SECTION 6.5: Fairness Audit Using SHAP

In [ ]:
# QUESTION: Does shelter_involvement contribute MORE to Bronx
# predictions than Staten Island predictions?
# If yes -> model treats same risk factor differently by borough -> bias

shap_df <- sv_importance(shp, kind='no')$data
# Alternative: extract SHAP matrix directly
shap_matrix <- shp$S  # n_cases x n_features matrix of SHAP values
colnames(shap_matrix) <- FEATURES

# Combine with borough info
shap_with_meta <- as_tibble(shap_matrix) %>%
  bind_cols(test %>% select(report_id, family_id)) %>%
  left_join(scr %>% select(family_id, borough) %>% distinct(),
            by='family_id') %>%
  filter(!is.na(borough))

# Average SHAP contribution per feature per borough
borough_shap <- shap_with_meta %>%
  group_by(borough) %>%
  summarise(across(all_of(FEATURES), mean, .names='shap_{.col}')) %>%
  pivot_longer(-borough, names_to='feature', values_to='avg_shap') %>%
  mutate(feature=str_remove(feature,'shap_'))

# Plot: does shelter contribute differently by borough?
borough_shap %>%
  filter(feature %in% c('shelter_involvement_flag','dv_history_flag',
                         'prior_reports_12mo','reporter_accuracy_score')) %>%
  ggplot(aes(x=borough, y=avg_shap, fill=avg_shap > 0)) +
  geom_col(show.legend=FALSE) +
  facet_wrap(~feature, scales='free_y') +
  coord_flip() +
  scale_fill_manual(values=c('TRUE'='#CB181D', 'FALSE'='#2171B5')) +
  labs(title='Average SHAP Contribution by Borough and Feature',
       subtitle='Equity flag: same feature contributing differently across boroughs',
       x=NULL, y='Average SHAP Value') +
  theme_minimal(base_size=11)

# INTERPRETATION:
# If shelter_involvement contributes more in Bronx than Staten Island
# the model may be penalizing families for being poor (housing instability)
# more in high-poverty boroughs
# Flag this for the ACS equity team before deployment

## SECTION 6.6: Counterfactual Explanations

In [ ]:
# Counterfactuals: 'What would need to change for this case to score below threshold?'
# More actionable than SHAP: tells caseworkers what to look for in intervention

# For the high-risk case: if prior_reports dropped from 4 to 1,
# what would the new score be?

high_risk_case <- X_test[high_risk_idx, , drop=FALSE]
current_score  <- predict(rf_model, high_risk_case)$predictions[,'1']

cat('Current case features:\n')
print(high_risk_case)
cat('Current risk score:', round(current_score, 3), '\n\n')

# Manual counterfactual: change one feature at a time
counterfactuals <- map_dfr(FEATURES, function(feat) {
  # Try setting each feature to its 25th percentile value
  modified <- high_risk_case
  modified[[feat]] <- quantile(df[[feat]], 0.25, na.rm=TRUE)
  new_score <- predict(rf_model, modified)$predictions[,'1']
  tibble(
    changed_feature = feat,
    original_value  = round(high_risk_case[[feat]], 3),
    changed_to      = round(modified[[feat]], 3),
    original_score  = round(current_score, 3),
    new_score       = round(new_score, 3),
    score_reduction = round(current_score - new_score, 3)
  )
}) %>%
  arrange(desc(score_reduction))

cat('If we could change ONE feature, which would reduce risk most?\n')
print(counterfactuals)

# CASEWORKER INTERPRETATION:
# 'If this family had no prior reports (counterfactual),
#  risk score drops from 0.87 to 0.54'
# This helps caseworkers understand what to monitor
# and what interventions might change the family's trajectory